In [6]:
import pandas as pd
import numpy as np
import re
import eurostat

In [7]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

### AI adoption rate (2021, 2023, 2024)

The dependent variable AI adoption rate is the proportion of firms (percent) withing each country - sector using at least one type of AI. Take it as it is. 

https://ec.europa.eu/eurostat/databrowser/product/page/isoc_eb_ain2

In [ ]:
ain2 = eurostat.get_data_df('isoc_eb_ain2')

In [9]:
print(ain2)

       freq size_emp nace_r2           indic_is    unit geo\TIME_PERIOD  2021  \
0         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              AT   NaN   
1         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              BA   NaN   
2         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              BE   NaN   
3         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              BG   NaN   
4         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              CY   NaN   
...     ...      ...     ...                ...     ...             ...   ...   
236938    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              RO   NaN   
236939    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              RS   NaN   
236940    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              SE   NaN   
236941    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              SI   NaN   
236942    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              SK   NaN   

         2023  2024   2025 

In [10]:
ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
ain2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236943 entries, 0 to 236942
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   freq      236943 non-null  object 
 1   size_emp  236943 non-null  object 
 2   nace_r2   236943 non-null  object 
 3   indic_is  236943 non-null  object 
 4   unit      236943 non-null  object 
 5   geo       236943 non-null  object 
 6   2021      109575 non-null  float64
 7   2023      162572 non-null  float64
 8   2024      137079 non-null  float64
 9   2025      170512 non-null  float64
dtypes: float64(4), object(6)
memory usage: 18.1+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_10484\2491334135.py:1: SyntaxWarning: invalid escape sequence '\T'
  ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [32]:
ain2.drop(['2021', '2023', '2024'], axis='columns', inplace=True)

In [11]:
ain2.describe()

,2021,2023,2024,2025
count,109575.000000,162572.000000,137079.000000,170512.000000
mean,10.853172,14.620912,14.050440,18.020146
std,21.003851,22.897424,20.980857,22.640605
min,0.000000,0.000000,0.000000,0.000000
25%,0.490000,1.200000,1.710000,3.100000
50%,2.410000,3.990000,5.240000,8.460000
75%,8.295000,16.250000,16.030000,22.800000
max,100.000000,100.000000,100.000000,100.000000


In [12]:
ain2.describe(include=["object", "bool"])

,freq,size_emp,nace_r2,indic_is,unit,geo
count,236943,236943,236943,236943,236943,236943
unique,1,1,50,63,6,36
top,A,GE10,C-E,E_AI_BINC,PC_ENT,PL
freq,236943,236943,4841,5359,95146,8138


In [14]:
ain2.to_csv('processed data/ain2.csv', index = False)

### Wages and Labour cost from LC survey (2016, 2020)

The wage variable measures wages, salaries, bonuses, allowances, employer contributions to saving schemes, and remuneration in kind, as received by workers, in euros per hour. 

Labor costs measure costs to firms, which include wage and non-wage costs minus subsidies. 

https://ec.europa.eu/eurostat/databrowser/view/lc_ncost_r2/default/table?lang=en

In [15]:
wg = pd.read_csv('data_panel/wage.csv')
print(wg)

     geo nace_r2  real_wage  year
0     AT       B  26.182355  2020
1     AT       B  25.838866  2021
2     AT       B  24.118279  2022
3     AT       B  24.309816  2023
4     AT       B  25.556963  2024
...   ..     ...        ...   ...
2200  SK       S   6.269014  2020
2201  SK       S   6.634986  2021
2202  SK       S   6.237505  2022
2203  SK       S   6.196412  2023
2204  SK       S   6.216386  2024

[2205 rows x 4 columns]


In [ ]:
# wg_split = wg[r'freq,currency,unit,sizeclas,nace_r2,lcstruct,geo\TIME_PERIOD'].str.split(',', expand=True)
# wg = pd.concat([wg, wg_split], axis=1)
# wg.rename(columns={0: 'freq', 1: 'currency', 2: 'unit',  3: 'sizeclas', 4: 'nace_r2', 5: 'lcstruct', 6:  'geo_TIME_PERIOD'}, inplace=True)
# wg = wg.drop(r'freq,currency,unit,sizeclas,nace_r2,lcstruct,geo\TIME_PERIOD', axis=1)
# wg.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
# wg.columns = wg.columns.str.strip()
# wg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254273 entries, 0 to 1254272
Data columns (total 11 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   2008      1254273 non-null  object
 1   2012      1254273 non-null  object
 2   2016      1254273 non-null  object
 3   2020      1254273 non-null  object
 4   freq      1254273 non-null  object
 5   currency  1254273 non-null  object
 6   unit      1254273 non-null  object
 7   sizeclas  1254273 non-null  object
 8   nace_r2   1254273 non-null  object
 9   lcstruct  1254273 non-null  object
 10  geo       1254273 non-null  object
dtypes: object(11)
memory usage: 105.3+ MB


In [ ]:
# years_take = ['2016', '2020']
# data_temp  = wg.copy()

# pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

# for year in years_take:
#     column_series = data_temp[year].astype(str).str.strip() 

#     extracted_df = column_series.str.extract(pattern, expand=True)
#     # print(extracted_df)

#     num_col_name = f'num_{year}'
#     data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
#     # print(ain2[num_col_name])
    
#     flag_col_name = f'flag_{year}'
#     data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
#     # print(ain2[flag_col_name])
#     # print('#########################')
# # Display the resulting DataFrame
# wg_new = data_temp
# print(wg_new)


                2008         2012         2016        2020 freq currency  \
0                 :         1.64         2.50        4.28     A      EUR   
1             22.37        27.20        30.99       33.00     A      EUR   
2                 :         3.71         3.91        4.94     A      EUR   
3               : @C       36.34        35.48       34.67     A      EUR   
4              2.53         3.42          : @C       5.87     A      EUR   
...              ...          ...          ...         ...  ...      ...   
1254268           :            :    31340162 d  32751700 d    A      PPS   
1254269   47753817 d           :    41270850 d  43110845 d    A      PPS   
1254270   34403143 d   35211330 d    33838580    36812309     A      PPS   
1254271           :            :    490463146   542511304     A      PPS   
1254272  5974985906   4267800190   4946674191           :     A      PPS   

            unit sizeclas nace_r2 lcstruct geo      num_2016 flag_2016  \
0        P_SA

In [ ]:
# wg_new.drop(['2008', '2012','2016', '2020'], axis='columns', inplace=True)
# wg_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254273 entries, 0 to 1254272
Data columns (total 11 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   freq       1254273 non-null  object 
 1   currency   1254273 non-null  object 
 2   unit       1254273 non-null  object 
 3   sizeclas   1254273 non-null  object 
 4   nace_r2    1254273 non-null  object 
 5   lcstruct   1254273 non-null  object 
 6   geo        1254273 non-null  object 
 7   num_2016   837744 non-null   float64
 8   flag_2016  1254273 non-null  object 
 9   num_2020   764253 non-null   float64
 10  flag_2020  1254273 non-null  object 
dtypes: float64(2), object(9)
memory usage: 105.3+ MB


In [ ]:
# wg_new.describe()

,num_2016,num_2020
count,8.377440e+05,7.642530e+05
mean,5.350333e+09,5.503536e+09
std,8.032602e+10,9.636738e+10
min,1.100000e-01,1.000000e+00
25%,4.620000e+02,6.220000e+02
50%,8.892000e+03,1.099200e+04
75%,1.584898e+06,2.209709e+06
max,1.262843e+13,1.710201e+13


In [ ]:
# wg_new.describe(include=["object", "bool"])

,2016,2020,freq,currency,unit,sizeclas,nace_r2,lcstruct,geo,flag_2016,flag_2020
count,1254273,1254273,1254273,1254273,1254273,1254273,1254273,1254273,1254273,1254273,1254273
unique,284120,265191,1,3,4,8,114,3,47,4,4
top,:,:,A,PPS,TOTAL,GE10,B-E,D01,EA16,NaN,NaN
freq,193731,281379,1254273,429552,330018,183378,12528,418250,32076,1141008,1163772


In [ ]:
# wg_new.to_csv('lc.csv', index = False)

### Concentration is the share of firms that have more than 250 employees. 

Possible dat from the 2005 - 2024. Selected 2016 and 2020. 

https://ec.europa.eu/eurostat/databrowser/view/sbs_sc_sca_r2/default/bar?lang=en

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/raw data/'

conctr = pd.read_csv(path + 'estat_sbs_sc_sca_r2.tsv/estat_sbs_sc_sca_r2.tsv', sep='\t')
print(conctr)

       freq,nace_r2,indic_sb,size_emp,geo\TIME_PERIOD 2005  2006  2007  2008   \
0                                   A,B,V11110,0-9,AL    :     :     :     :    
1                                   A,B,V11110,0-9,AT    :     :     :   237    
2                                   A,B,V11110,0-9,BA    :     :     :     :    
3                                   A,B,V11110,0-9,BE    :     :     :   : @C   
4                                   A,B,V11110,0-9,BG    :     :     :   199    
...                                               ...   ...   ...   ...   ...   
483382                          A,S95,V92100,TOTAL,RS    :     :     :     :    
483383                          A,S95,V92100,TOTAL,SE    :     :     :     :    
483384                          A,S95,V92100,TOTAL,SI    :     :     :     :    
483385                          A,S95,V92100,TOTAL,SK    :     :     :     :    
483386                          A,S95,V92100,TOTAL,UK    :     :     :     :    

       2009  2010   2011   

In [56]:
conctr_split = conctr[r'freq,nace_r2,indic_sb,size_emp,geo\TIME_PERIOD'].str.split(',', expand=True)
conctr = pd.concat([conctr, conctr_split], axis=1)
conctr.rename(columns={0: 'freq', 1: 'nace_r2', 2: 'indic_sb',  3: 'size_emp', 4: 'geo_TIME_PERIOD'}, inplace=True)
conctr = conctr.drop(r'freq,nace_r2,indic_sb,size_emp,geo\TIME_PERIOD', axis=1)
conctr.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
conctr.columns = conctr.columns.str.strip()
conctr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483387 entries, 0 to 483386
Data columns (total 21 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   2005      483387 non-null  object
 1   2006      483387 non-null  object
 2   2007      483387 non-null  object
 3   2008      483387 non-null  object
 4   2009      483387 non-null  object
 5   2010      483387 non-null  object
 6   2011      483387 non-null  object
 7   2012      483387 non-null  object
 8   2013      483387 non-null  object
 9   2014      483387 non-null  object
 10  2015      483387 non-null  object
 11  2016      483387 non-null  object
 12  2017      483387 non-null  object
 13  2018      483387 non-null  object
 14  2019      483387 non-null  object
 15  2020      483387 non-null  object
 16  freq      483387 non-null  object
 17  nace_r2   483387 non-null  object
 18  indic_sb  483387 non-null  object
 19  size_emp  483387 non-null  object
 20  geo       483387 non-null 

In [57]:
years_take = ['2016', '2020']
data_temp  = conctr.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
conctr_new = data_temp
print(conctr_new)


       2005 2006 2007  2008  2009  2010   2011   2012   2013   2014  ...  \
0        :    :    :     :     :     :      :      :      :      :   ...   
1        :    :    :   237   244   245    242    238    238    227   ...   
2        :    :    :     :     :     :   150 p  121 p  118 p  121 p  ...   
3        :    :    :   : @C  248   249    227    160    175    149   ...   
4        :    :    :   199   230   241    245    252    243    228   ...   
...     ...  ...  ...   ...   ...   ...    ...    ...    ...    ...  ...   
483382   :    :    :     :     :     :      :      :      :      :   ...   
483383   :    :    :     :     :     :      :      :      :      :   ...   
483384   :    :    :     :     :     :      :      :      :      :   ...   
483385   :    :    :     :     :     :      :      :      :    1.3   ...   
483386   :    :    :     :     :     :      :      :      :      :   ...   

        2020 freq nace_r2 indic_sb size_emp geo num_2016 flag_2016 num_2020  \
0       

In [58]:
y = 2005
while y <= 2020:
    column_to_drop = str(y)
    conctr_new.drop([column_to_drop], axis='columns', inplace=True)
    y += 1
conctr_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483387 entries, 0 to 483386
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   freq       483387 non-null  object 
 1   nace_r2    483387 non-null  object 
 2   indic_sb   483387 non-null  object 
 3   size_emp   483387 non-null  object 
 4   geo        483387 non-null  object 
 5   num_2016   221352 non-null  float64
 6   flag_2016  483387 non-null  object 
 7   num_2020   214875 non-null  float64
 8   flag_2020  483387 non-null  object 
dtypes: float64(2), object(7)
memory usage: 33.2+ MB


In [60]:
conctr_new.to_csv('conctr.csv', index = False)

### Proportion of firms using computer

Possible data from 2009 - 2024. Selected 2016 and 2020. 

https://ec.europa.eu/eurostat/databrowser/view/isoc_ci_cm_pn2/default/table?lang=en

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

comp = pd.read_csv(path + 'estat_isoc_ci_cm_pn2.tsv/estat_isoc_ci_cm_pn2.tsv', sep='\t')
print(comp)

     freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD   2009    2010   \
0                             A,GE10,C,P_CUSE,PC_EMP,AT   51.35   53.31    
1                             A,GE10,C,P_CUSE,PC_EMP,BA       :       :    
2                             A,GE10,C,P_CUSE,PC_EMP,BE      : u  61.47    
3                             A,GE10,C,P_CUSE,PC_EMP,BG    9.58   16.11    
4                             A,GE10,C,P_CUSE,PC_EMP,CY   25.17   25.81    
...                                                 ...      ...     ...   
8815                  A,GE10,S951,P_IUSE,PC_EMP_IACC,SE       :       :    
8816                  A,GE10,S951,P_IUSE,PC_EMP_IACC,SI       :       :    
8817                  A,GE10,S951,P_IUSE,PC_EMP_IACC,SK       :       :    
8818                  A,GE10,S951,P_IUSE,PC_EMP_IACC,TR       :       :    
8819                  A,GE10,S951,P_IUSE,PC_EMP_IACC,UK       :       :    

       2011     2012     2013     2014     2015    2016    2017    2018   \
0         :

In [5]:
comp_split = comp [r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
comp = pd.concat([comp, comp_split], axis=1)
comp.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is', 4: 'unit', 5: 'geo\TIME_PERIOD'}, inplace=True)
comp  = comp.drop(r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD', axis=1)
comp.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
comp.columns = comp.columns.str.strip()
comp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8820 entries, 0 to 8819
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   2009             8820 non-null   object
 1   2010             8820 non-null   object
 2   2011             8820 non-null   object
 3   2012             8820 non-null   object
 4   2013             8820 non-null   object
 5   2014             8820 non-null   object
 6   2015             8820 non-null   object
 7   2016             8820 non-null   object
 8   2017             8820 non-null   object
 9   2018             8820 non-null   object
 10  2019             8820 non-null   object
 11  2020             8820 non-null   object
 12  2021             8820 non-null   object
 13  2022             8820 non-null   object
 14  2023             8820 non-null   object
 15  2024             8820 non-null   object
 16  freq             8820 non-null   object
 17  size_emp         8820 non-null   

<>:3: SyntaxWarning: invalid escape sequence '\T'
<>:3: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_35152\2602886828.py:3: SyntaxWarning: invalid escape sequence '\T'
  comp.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is', 4: 'unit', 5: 'geo\TIME_PERIOD'}, inplace=True)


In [6]:
years_take = ['2016', '2020']
data_temp  = comp.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
comp_new = data_temp
print(comp_new)


        2009    2010    2011     2012     2013     2014     2015    2016  \
0     51.35   53.31       :    54.27    55.19    56.85        :       :    
1         :       :       :        :        :        :        :       :    
2        : u  61.47   62.10    60.22       : u      : u   56.56   57.93    
3      9.58   16.11   16.53    16.39    17.51    18.38    18.38   19.17    
4     25.17   25.81   26.27    30.44    28.65    35.56    30.59   32.96    
...      ...     ...     ...      ...      ...      ...      ...     ...   
8815      :       :       :    98.35   100.00    97.79   100.00       :    
8816      :       :       :   100.00   100.00   100.00   100.00       :    
8817      :       :       :    82.88    83.96    74.63   100.00       :    
8818      :       :       :        :        :        :    70.61       :    
8819      :       :       :    91.30    88.18    93.61    91.41       :    

        2017    2018  ... freq size_emp nace_r2 indic_is         unit  \
0         :   

In [7]:
y = 2009
while y <= 2024:
    column_to_drop = str(y)
    comp_new.drop([column_to_drop], axis='columns', inplace=True)
    y += 1
comp_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8820 entries, 0 to 8819
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   freq             8820 non-null   object 
 1   size_emp         8820 non-null   object 
 2   nace_r2          8820 non-null   object 
 3   indic_is         8820 non-null   object 
 4   unit             8820 non-null   object 
 5   geo\TIME_PERIOD  8820 non-null   object 
 6   num_2016         3184 non-null   float64
 7   flag_2016        8820 non-null   object 
 8   num_2020         1153 non-null   float64
 9   flag_2020        8820 non-null   object 
dtypes: float64(2), object(8)
memory usage: 689.2+ KB


In [8]:
comp_new.to_csv('comp.csv', index = False)

### Cost  capital 

is the harmonized bank lending rate on outstanding loans with more than 5 years' maturity; it varies across countries only. Cost of capital gathered in other notebook. 

Data source - European Central Band db

### Energy cost 

Possible data from 2007 - 2025. Selected 2016 and 2020. 


https://ec.europa.eu/eurostat/databrowser/view/nrg_pc_205/default/line?lang=en

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/raw data/'

enrg_cst = pd.read_csv(path + 'estat_nrg_pc_205.tsv/estat_nrg_pc_205.tsv', sep='\t')
print(enrg_cst)

     freq,product,nrg_cons,unit,tax,currency,geo\TIME_PERIOD 2007-S1   \
0                     S,6000,MWH20-499,KWH,I_TAX,EUR,AL            :    
1                     S,6000,MWH20-499,KWH,I_TAX,EUR,AT            :    
2                     S,6000,MWH20-499,KWH,I_TAX,EUR,BA            :    
3                     S,6000,MWH20-499,KWH,I_TAX,EUR,BE            :    
4                     S,6000,MWH20-499,KWH,I_TAX,EUR,BG            :    
...                                                 ...           ...   
2881                    S,6000,TOT_KWH,KWH,X_VAT,PPS,RS            :    
2882                    S,6000,TOT_KWH,KWH,X_VAT,PPS,SE            :    
2883                    S,6000,TOT_KWH,KWH,X_VAT,PPS,SI            :    
2884                    S,6000,TOT_KWH,KWH,X_VAT,PPS,SK            :    
2885                    S,6000,TOT_KWH,KWH,X_VAT,PPS,TR            :    

     2007-S2  2008-S1  2008-S2  2009-S1  2009-S2  2010-S1  2010-S2  2011-S1   \
0          :        :        :        :    

In [8]:
enrg_cst_split = enrg_cst[r'freq,product,nrg_cons,unit,tax,currency,geo\TIME_PERIOD'].str.split(',', expand=True)
enrg_cst = pd.concat([enrg_cst, enrg_cst_split], axis=1)
enrg_cst.rename(columns={0: 'freq', 1: 'product', 2: 'nrg_cons',  3: 'unit', 4: 'tax', 5: 'currency', 6:'geo_TIME_PERIOD'}, inplace=True)
enrg_cst = enrg_cst.drop(r'freq,product,nrg_cons,unit,tax,currency,geo\TIME_PERIOD', axis=1)
enrg_cst.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
enrg_cst.columns = enrg_cst.columns.str.strip()
enrg_cst.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2886 entries, 0 to 2885
Data columns (total 44 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2007-S1   2886 non-null   object
 1   2007-S2   2886 non-null   object
 2   2008-S1   2886 non-null   object
 3   2008-S2   2886 non-null   object
 4   2009-S1   2886 non-null   object
 5   2009-S2   2886 non-null   object
 6   2010-S1   2886 non-null   object
 7   2010-S2   2886 non-null   object
 8   2011-S1   2886 non-null   object
 9   2011-S2   2886 non-null   object
 10  2012-S1   2886 non-null   object
 11  2012-S2   2886 non-null   object
 12  2013-S1   2886 non-null   object
 13  2013-S2   2886 non-null   object
 14  2014-S1   2886 non-null   object
 15  2014-S2   2886 non-null   object
 16  2015-S1   2886 non-null   object
 17  2015-S2   2886 non-null   object
 18  2016-S1   2886 non-null   object
 19  2016-S2   2886 non-null   object
 20  2017-S1   2886 non-null   object
 21  2017-S2   2886

In [9]:
years_take = ['2016-S1', '2016-S2', '2020-S1', '2020-S2']

data_temp  = enrg_cst.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
enrg_cst_new = data_temp
print(enrg_cst_new)


     2007-S1  2007-S2  2008-S1  2008-S2  2009-S1  2009-S2  2010-S1  2010-S2  \
0         :        :        :        :        :        :        :        :    
1         :   0.1412   0.1476    0.144   0.1601   0.1588   0.1545   0.1547    
2         :        :        :        :        :        :        :        :    
3         :   0.1435   0.1706   0.1728   0.1605   0.1547   0.1562   0.1584    
4         :   0.0767   0.0767   0.0874   0.0879   0.0864   0.0854   0.0851    
...      ...      ...      ...      ...      ...      ...      ...      ...   
2881      :        :        :        :        :        :        :        :    
2882      :        :        :        :        :        :        :        :    
2883      :        :        :        :        :        :        :        :    
2884      :        :        :        :        :        :        :        :    
2885      :        :        :        :        :        :        :        :    

      2011-S1  2011-S2  ... currency geo num_2016-S

In [10]:
start_year = 2005
end_year = 2025
seasons = ['S1', 'S2']

years_to_drop = []
for y in range(start_year, end_year + 1):
    for s in seasons:
        years_to_drop.append(f'{y}-{s}')

existing_cols_to_drop = [col for col in years_to_drop if col in enrg_cst_new.columns]
enrg_cst_new.drop(existing_cols_to_drop, axis='columns', inplace=True)
enrg_cst_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2886 entries, 0 to 2885
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   freq          2886 non-null   object 
 1   product       2886 non-null   object 
 2   nrg_cons      2886 non-null   object 
 3   unit          2886 non-null   object 
 4   tax           2886 non-null   object 
 5   currency      2886 non-null   object 
 6   geo           2886 non-null   object 
 7   num_2016-S1   2115 non-null   float64
 8   flag_2016-S1  2886 non-null   object 
 9   num_2016-S2   2106 non-null   float64
 10  flag_2016-S2  2886 non-null   object 
 11  num_2020-S1   2358 non-null   float64
 12  flag_2020-S1  2886 non-null   object 
 13  num_2020-S2   2304 non-null   float64
 14  flag_2020-S2  2886 non-null   object 
dtypes: float64(4), object(11)
memory usage: 338.3+ KB


In [11]:
enrg_cst_new['2016'] = (enrg_cst_new['num_2016-S1'] + enrg_cst_new['num_2016-S2'])/2
enrg_cst_new['2020'] = (enrg_cst_new['num_2020-S1'] + enrg_cst_new['num_2020-S2'])/2
print(enrg_cst_new)

     freq product   nrg_cons unit    tax currency geo  num_2016-S1  \
0       S    6000  MWH20-499  KWH  I_TAX      EUR  AL          NaN   
1       S    6000  MWH20-499  KWH  I_TAX      EUR  AT       0.1519   
2       S    6000  MWH20-499  KWH  I_TAX      EUR  BA       0.1003   
3       S    6000  MWH20-499  KWH  I_TAX      EUR  BE       0.1803   
4       S    6000  MWH20-499  KWH  I_TAX      EUR  BG       0.1285   
...   ...     ...        ...  ...    ...      ...  ..          ...   
2881    S    6000    TOT_KWH  KWH  X_VAT      PPS  RS          NaN   
2882    S    6000    TOT_KWH  KWH  X_VAT      PPS  SE          NaN   
2883    S    6000    TOT_KWH  KWH  X_VAT      PPS  SI          NaN   
2884    S    6000    TOT_KWH  KWH  X_VAT      PPS  SK          NaN   
2885    S    6000    TOT_KWH  KWH  X_VAT      PPS  TR          NaN   

     flag_2016-S1  num_2016-S2 flag_2016-S2  num_2020-S1 flag_2020-S1  \
0             NaN          NaN          NaN       0.1245            e   
1            

In [12]:
import numpy as np

enrg_cst_new['flag_2016'] = np.where(
    enrg_cst_new['flag_2016-S2'].notna(), 
    enrg_cst_new['flag_2016-S2'], 
    enrg_cst_new['flag_2016-S1']
)

enrg_cst_new['flag_2020'] = np.where(
    enrg_cst_new['flag_2020-S2'].notna(), 
    enrg_cst_new['flag_2020-S2'], 
    enrg_cst_new['flag_2020-S1']
)

print(enrg_cst_new)

     freq product   nrg_cons unit    tax currency geo  num_2016-S1  \
0       S    6000  MWH20-499  KWH  I_TAX      EUR  AL          NaN   
1       S    6000  MWH20-499  KWH  I_TAX      EUR  AT       0.1519   
2       S    6000  MWH20-499  KWH  I_TAX      EUR  BA       0.1003   
3       S    6000  MWH20-499  KWH  I_TAX      EUR  BE       0.1803   
4       S    6000  MWH20-499  KWH  I_TAX      EUR  BG       0.1285   
...   ...     ...        ...  ...    ...      ...  ..          ...   
2881    S    6000    TOT_KWH  KWH  X_VAT      PPS  RS          NaN   
2882    S    6000    TOT_KWH  KWH  X_VAT      PPS  SE          NaN   
2883    S    6000    TOT_KWH  KWH  X_VAT      PPS  SI          NaN   
2884    S    6000    TOT_KWH  KWH  X_VAT      PPS  SK          NaN   
2885    S    6000    TOT_KWH  KWH  X_VAT      PPS  TR          NaN   

     flag_2016-S1  num_2016-S2 flag_2016-S2  num_2020-S1 flag_2020-S1  \
0             NaN          NaN          NaN       0.1245            e   
1            

In [ ]:
enrg_cst_new = enrg_cst_new.drop(columns=['flag_2016-S1', 'flag_2016-S2', 'flag_2020-S1', 'flag_2020-S2', 'num_2016-S1', 'num_2016-S2', 'num_2020-S1', 'num_2020-S2'])

print(enrg_cst_new)

     freq product   nrg_cons unit    tax currency geo     2016     2020  \
0       S    6000  MWH20-499  KWH  I_TAX      EUR  AL      NaN  0.12440   
1       S    6000  MWH20-499  KWH  I_TAX      EUR  AT  0.15045  0.16190   
2       S    6000  MWH20-499  KWH  I_TAX      EUR  BA  0.10040  0.10610   
3       S    6000  MWH20-499  KWH  I_TAX      EUR  BE  0.18490  0.18465   
4       S    6000  MWH20-499  KWH  I_TAX      EUR  BG  0.12105  0.11335   
...   ...     ...        ...  ...    ...      ...  ..      ...      ...   
2881    S    6000    TOT_KWH  KWH  X_VAT      PPS  RS      NaN      NaN   
2882    S    6000    TOT_KWH  KWH  X_VAT      PPS  SE      NaN  0.04425   
2883    S    6000    TOT_KWH  KWH  X_VAT      PPS  SI      NaN      NaN   
2884    S    6000    TOT_KWH  KWH  X_VAT      PPS  SK      NaN      NaN   
2885    S    6000    TOT_KWH  KWH  X_VAT      PPS  TR      NaN      NaN   

     flag_2016 flag_2020  
0          NaN         e  
1          NaN       NaN  
2          NaN    

In [17]:
enrg_cst_new.to_csv('enrg_cst.csv', index = False)

### Human capital 

Possible data from 2008 - 2026. Selected 2016 and 2020. 

https://ec.europa.eu/eurostat/databrowser/view/edat_lfs_9910__custom_9652224/default/table?lang=en

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/raw data/'

hum_cap = pd.read_csv(path + 'estat_edat_lfs_9910.tsv/estat_edat_lfs_9910.tsv', sep='\t')
print(hum_cap)

       freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD 2008  2009     2010   \
0                               A,PC,A,ED0-2,Y15-24,F,AT   : u   : u      : u   
1                               A,PC,A,ED0-2,Y15-24,F,BA    :     :        :    
2                               A,PC,A,ED0-2,Y15-24,F,BE  : bu   : u      : u   
3                               A,PC,A,ED0-2,Y15-24,F,BG   : u   : u     : bu   
4                               A,PC,A,ED0-2,Y15-24,F,CH   : u   : u  68.1 bu   
...                                                  ...   ...   ...      ...   
241725                          A,PC,U,ED5-8,Y55-74,T,SE   : u   : u      : u   
241726                          A,PC,U,ED5-8,Y55-74,T,SI    :     :        :    
241727                          A,PC,U,ED5-8,Y55-74,T,SK   : u    :        :    
241728                          A,PC,U,ED5-8,Y55-74,T,TR    :    : u      : u   
241729                          A,PC,U,ED5-8,Y55-74,T,UK  : bu   : u     : bu   

         2011    2012    20

In [23]:
hum_cap_split = hum_cap[r'freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD'].str.split(',', expand=True)
hum_cap = pd.concat([hum_cap, hum_cap_split], axis=1)
hum_cap.rename(columns={0: 'freq', 1: 'unit', 2: 'nace_r2',  3: 'isced11', 4: 'age', 5 : 'sex', 6 : 'geo\TIME_PERIOD'}, inplace=True)
hum_cap  = hum_cap.drop(r'freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD', axis=1)
hum_cap.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
hum_cap.columns = hum_cap.columns.str.strip()
hum_cap.info()

<>:3: SyntaxWarning: invalid escape sequence '\T'
<>:3: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_20564\889841396.py:3: SyntaxWarning: invalid escape sequence '\T'
  hum_cap.rename(columns={0: 'freq', 1: 'unit', 2: 'nace_r2',  3: 'isced11', 4: 'age', 5 : 'sex', 6 : 'geo\TIME_PERIOD'}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241730 entries, 0 to 241729
Data columns (total 24 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   2008             241730 non-null  object
 1   2009             241730 non-null  object
 2   2010             241730 non-null  object
 3   2011             241730 non-null  object
 4   2012             241730 non-null  object
 5   2013             241730 non-null  object
 6   2014             241730 non-null  object
 7   2015             241730 non-null  object
 8   2016             241730 non-null  object
 9   2017             241730 non-null  object
 10  2018             241730 non-null  object
 11  2019             241730 non-null  object
 12  2020             241730 non-null  object
 13  2021             241730 non-null  object
 14  2022             241730 non-null  object
 15  2023             241730 non-null  object
 16  2024             241730 non-null  object
 17  freq      

In [24]:
years_take = ['2016', '2020']
data_temp  = hum_cap.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
hum_cap_new = data_temp
print(hum_cap_new)

        2008 2009     2010    2011    2012    2013     2014    2015    2016  \
0        : u  : u      : u     : u     : u     : u     : bu     : u     : u   
1         :    :        :       :       :       :        :       :       :    
2       : bu  : u      : u     : u     : u     : u     : bu     : u     : u   
3        : u  : u     : bu    : bu     : u     : u     : bu     : u     : u   
4        : u  : u  68.1 bu  60.4 u  46.9 u  58.9 u  61.0 bu  36.8 u  33.1 u   
...      ...  ...      ...     ...     ...     ...      ...     ...     ...   
241725   : u  : u      : u     : u     : u     : u     : bu     : u     : u   
241726    :    :        :       :       :       :        :       :       :    
241727   : u   :        :     : bu     : u      :      : bu     : u     : u   
241728    :   : u      : u  72.4 u  74.7 u  36.0 u       :      : u     : u   
241729  : bu  : u     : bu    : bu     : u     : u     : bu     : u     : u   

          2017  ... unit nace_r2 isced11     age se

In [25]:
y = 2008
while y <= 2024:
    column_to_drop = str(y)
    hum_cap_new.drop([column_to_drop], axis='columns', inplace=True)
    y += 1
hum_cap_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241730 entries, 0 to 241729
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   freq             241730 non-null  object 
 1   unit             241730 non-null  object 
 2   nace_r2          241730 non-null  object 
 3   isced11          241730 non-null  object 
 4   age              241730 non-null  object 
 5   sex              241730 non-null  object 
 6   geo\TIME_PERIOD  241730 non-null  object 
 7   num_2016         124626 non-null  float64
 8   flag_2016        241730 non-null  object 
 9   num_2020         114497 non-null  float64
 10  flag_2020        241730 non-null  object 
dtypes: float64(2), object(9)
memory usage: 20.3+ MB


In [26]:
hum_cap_new.to_csv('human_capital.csv')

### Current Wages 

Wages per hour (retrieved from the Labour cost level dataset (Labour Survey * Labour Cost Index))

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'


lc = pd.read_csv(path + 'estat_lc_lci_lev.tsv/estat_lc_lci_lev.tsv', sep='\t')
print(lc)

      freq,unit,lcstruct,nace_r2,geo\TIME_PERIOD  2008   2012   2016   2020   \
0                                 A,EUR,D11,B,AL     :    2.1    2.7      :    
1                                 A,EUR,D11,B,AT  22.2   26.9   28.1   28.4    
2                                 A,EUR,D11,B,BA     :    6.3    6.7      :    
3                                 A,EUR,D11,B,BE     :   30.3   29.5   31.3    
4                                 A,EUR,D11,B,BG   3.3      5    5.7    8.1    
...                                          ...    ...    ...    ...    ...   
11070                A,RT_PRE_NAC,D1_D4_MD5,S,RO     :      :      :    8.4    
11071                A,RT_PRE_NAC,D1_D4_MD5,S,RS     :      :      :    6.6    
11072                A,RT_PRE_NAC,D1_D4_MD5,S,SE     :      :      :    0.1    
11073                A,RT_PRE_NAC,D1_D4_MD5,S,SI     :      :      :    4.1    
11074                A,RT_PRE_NAC,D1_D4_MD5,S,SK     :      :      :    4.1    

       2021   2022    2023   2024   
0 

In [18]:
lc_split = lc[r'freq,unit,lcstruct,nace_r2,geo\TIME_PERIOD'].str.split(',', expand=True)
lc = pd.concat([lc, lc_split], axis=1)
lc.rename(columns={0: 'freq', 1: 'unit', 2: 'lcstruct',  3: 'nace_r2',  4:  'geo_TIME_PERIOD'}, inplace=True)
lc = lc.drop(r'freq,unit,lcstruct,nace_r2,geo\TIME_PERIOD', axis=1)
lc.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
lc.columns = lc.columns.str.strip()
lc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11075 entries, 0 to 11074
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2008      11075 non-null  object
 1   2012      11075 non-null  object
 2   2016      11075 non-null  object
 3   2020      11075 non-null  object
 4   2021      11075 non-null  object
 5   2022      11075 non-null  object
 6   2023      11075 non-null  object
 7   2024      11075 non-null  object
 8   freq      11075 non-null  object
 9   unit      11075 non-null  object
 10  lcstruct  11075 non-null  object
 11  nace_r2   11075 non-null  object
 12  geo       11075 non-null  object
dtypes: object(13)
memory usage: 1.1+ MB


In [19]:
years_take = ['2020', '2021', '2022', '2023', '2024']
data_temp  = lc.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
lc_new = data_temp
print(lc_new)


        2008   2012   2016   2020   2021   2022    2023   2024 freq  \
0         :    2.1    2.7      :      :      :       :      :     A   
1      22.2   26.9   28.1   28.4   28.8   29.2   31.7 p  34.3     A   
2         :    6.3    6.7      :      :      :       :      :     A   
3         :   30.3   29.5   31.3   31.7     34    36.7   37.6     A   
4       3.3      5    5.7    8.1    8.8   10.3    11.4   12.2     A   
...      ...    ...    ...    ...    ...    ...     ...    ...  ...   
11070     :      :      :    8.4    4.8   19.8     8.2   18.9     A   
11071     :      :      :    6.6    9.5   15.3    12.2   13.3     A   
11072     :      :      :    0.1      4      1     3.8    2.2     A   
11073     :      :      :    4.1    3.6    4.2      14      7     A   
11074     :      :      :    4.1    5.5   12.7    11.1    6.4     A   

             unit  ... num_2020 flag_2020 num_2021  flag_2021 num_2022  \
0             EUR  ...      NaN       NaN      NaN        NaN      NaN   

In [20]:
lc_new.drop(['2008', '2012','2016'], axis='columns', inplace=True)
lc_new.drop(years_take, axis='columns', inplace=True)
lc_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11075 entries, 0 to 11074
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   freq       11075 non-null  object 
 1   unit       11075 non-null  object 
 2   lcstruct   11075 non-null  object 
 3   nace_r2    11075 non-null  object 
 4   geo        11075 non-null  object 
 5   num_2020   8758 non-null   float64
 6   flag_2020  11075 non-null  object 
 7   num_2021   8802 non-null   float64
 8   flag_2021  11075 non-null  object 
 9   num_2022   9272 non-null   float64
 10  flag_2022  11075 non-null  object 
 11  num_2023   9461 non-null   float64
 12  flag_2023  11075 non-null  object 
 13  num_2024   9211 non-null   float64
 14  flag_2024  11075 non-null  object 
dtypes: float64(5), object(10)
memory usage: 1.3+ MB


In [21]:
lc_new.to_csv('lc_test.csv', index = False)

### Enterprises that provided training to develop/upgrade ICT skills of their personnel by NACE activity

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

train = pd.read_csv(path + 'estat_isoc_ske_ittn2.tsv/estat_isoc_ske_ittn2.tsv', sep='\t')
print(train)

      freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD   2012    2014   \
0                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,AT       :       :    
1                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,BA       :       :    
2                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,BE       :       :    
3                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,BG       :       :    
4                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,CY       :       :    
...                                                  ...      ...     ...   
16549                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,SE   37.50      : u   
16550                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,SI   57.14   49.93    
16551                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,SK   11.61   57.50    
16552                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,TR       :       :    
16553                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,UK   46.48   43.08    

        2015    2016    2017    2018     2019    2020   2022    2024   
0  

In [5]:
train_split = train[r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
train = pd.concat([train, train_split], axis=1)
train.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is',  4: 'unit', 5:  'geo_TIME_PERIOD'}, inplace=True)
train = train.drop(r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD', axis=1)
train.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
train.columns = train.columns.str.strip()
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16554 entries, 0 to 16553
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2012      16554 non-null  object
 1   2014      16554 non-null  object
 2   2015      16554 non-null  object
 3   2016      16554 non-null  object
 4   2017      16554 non-null  object
 5   2018      16554 non-null  object
 6   2019      16554 non-null  object
 7   2020      16554 non-null  object
 8   2022      16554 non-null  object
 9   2024      16554 non-null  object
 10  freq      16554 non-null  object
 11  size_emp  16554 non-null  object
 12  nace_r2   16554 non-null  object
 13  indic_is  16554 non-null  object
 14  unit      16554 non-null  object
 15  geo       16554 non-null  object
dtypes: object(16)
memory usage: 2.0+ MB


In [7]:
years_take = ['2020', '2022', '2024']
data_temp  = train.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
train_new = data_temp
print(train_new)

         2012    2014    2015    2016    2017    2018     2019    2020   2022  \
0          :       :       :   17.86   14.70   14.47     7.07    3.96   5.50    
1          :       :       :       :       :      : u      : u   2.90   7.08    
2          :       :       :      : u     : u     : u      : u     : u  8.87    
3          :       :       :      : u   1.49    1.30       : u     : u  1.80    
4          :       :       :    3.54   10.23   11.55    12.07   11.28   8.62    
...       ...     ...     ...     ...     ...     ...      ...     ...    ...   
16549  37.50      : u  27.27   50.00   40.91   31.03   33.33 b      :      :    
16550  57.14   49.93     : @C     : u    : @C    : @C     : @C      :      :    
16551  11.61   57.50       0   46.88   21.15       0        0       :      :    
16552      :       :      : u      :       :   39.56    28.05       :      :    
16553  46.48   43.08   55.27   49.64      : u     : u      : u      :      :    

         2024  ... nace_r2 

In [8]:
train_new.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
train_new.drop(years_take, axis='columns', inplace=True)
train_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16554 entries, 0 to 16553
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   freq       16554 non-null  object 
 1   size_emp   16554 non-null  object 
 2   nace_r2    16554 non-null  object 
 3   indic_is   16554 non-null  object 
 4   unit       16554 non-null  object 
 5   geo        16554 non-null  object 
 6   num_2020   5499 non-null   float64
 7   flag_2020  16554 non-null  object 
 8   num_2022   7014 non-null   float64
 9   flag_2022  16554 non-null  object 
 10  num_2024   7069 non-null   float64
 11  flag_2024  16554 non-null  object 
dtypes: float64(3), object(9)
memory usage: 1.5+ MB


In [9]:
train_new.to_csv('train.csv', index = False)

### Enterproces that employ ICT specialists by NACE Rev.2 activity


In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

ict_spec = pd.read_csv(path + 'estat_isoc_ske_itspen2.tsv/estat_isoc_ske_itspen2.tsv', sep='\t')
print(ict_spec)

     freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD   2012    2014   \
0                            A,GE10,C,E_ITSP2,PC_ENT,AL       :       :    
1                            A,GE10,C,E_ITSP2,PC_ENT,AT   35.12   31.01    
2                            A,GE10,C,E_ITSP2,PC_ENT,BA       :       :    
3                            A,GE10,C,E_ITSP2,PC_ENT,BE   32.85      : u   
4                            A,GE10,C,E_ITSP2,PC_ENT,BG   10.34   17.20    
...                                                 ...      ...     ...   
3338                 A,GE10,S951,E_ITSP2,PC_ENT_CUSE,SE   75.00      : u   
3339                 A,GE10,S951,E_ITSP2,PC_ENT_CUSE,SI   71.43   83.33    
3340                 A,GE10,S951,E_ITSP2,PC_ENT_CUSE,SK   41.96   57.50    
3341                 A,GE10,S951,E_ITSP2,PC_ENT_CUSE,TR       :       :    
3342                 A,GE10,S951,E_ITSP2,PC_ENT_CUSE,UK   66.67   65.38    

        2015    2016    2017    2018    2019    2020     2022    2024   
0          :  

In [11]:
ict_spec_split = ict_spec[r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
ict_spec = pd.concat([ict_spec, ict_spec_split], axis=1)
ict_spec.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is',  4: 'unit', 5:  'geo_TIME_PERIOD'}, inplace=True)
ict_spec = ict_spec.drop(r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD', axis=1)
ict_spec.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
ict_spec.columns = ict_spec.columns.str.strip()
ict_spec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3343 entries, 0 to 3342
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2012      3343 non-null   object
 1   2014      3343 non-null   object
 2   2015      3343 non-null   object
 3   2016      3343 non-null   object
 4   2017      3343 non-null   object
 5   2018      3343 non-null   object
 6   2019      3343 non-null   object
 7   2020      3343 non-null   object
 8   2022      3343 non-null   object
 9   2024      3343 non-null   object
 10  freq      3343 non-null   object
 11  size_emp  3343 non-null   object
 12  nace_r2   3343 non-null   object
 13  indic_is  3343 non-null   object
 14  unit      3343 non-null   object
 15  geo       3343 non-null   object
dtypes: object(16)
memory usage: 418.0+ KB


In [12]:
years_take = ['2020', '2022', '2024']
data_temp  = ict_spec.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
ict_spec_new = data_temp
print(ict_spec_new)

        2012    2014     2015    2016    2017    2018    2019    2020  \
0         :       :        :       :       :       :       :       :    
1     35.12   31.01    31.44   35.22   30.53   28.03   29.00   28.74    
2         :       :        :       :       :   12.50   13.33   12.92    
3     32.85      : u      : u     : u     : u     : u  30.43      : u   
4     10.34   17.20    16.76   15.71   17.61   14.58   16.27   15.16    
...      ...     ...      ...     ...     ...     ...     ...     ...   
3338  75.00      : u   72.73   58.33   65.91      : u  39.29       :    
3339  71.43   83.33    83.33      : u    : @C    : @C     : u      :    
3340  41.96   57.50   100.00   93.75   28.85   28.85   31.82       :    
3341      :       :       : u      :       :   63.22   60.43       :    
3342  66.67   65.38    78.18   79.93      : u     : u     : u      :    

         2022    2024  ... nace_r2 indic_is         unit geo num_2020  \
0     11.22 b      :   ...       C  E_ITSP2       

In [13]:
ict_spec_new.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
ict_spec_new.drop(years_take, axis='columns', inplace=True)
ict_spec_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3343 entries, 0 to 3342
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   freq       3343 non-null   object 
 1   size_emp   3343 non-null   object 
 2   nace_r2    3343 non-null   object 
 3   indic_is   3343 non-null   object 
 4   unit       3343 non-null   object 
 5   geo        3343 non-null   object 
 6   num_2020   1137 non-null   float64
 7   flag_2020  3343 non-null   object 
 8   num_2022   1426 non-null   float64
 9   flag_2022  3343 non-null   object 
 10  num_2024   1427 non-null   float64
 11  flag_2024  3343 non-null   object 
dtypes: float64(3), object(9)
memory usage: 313.5+ KB


In [14]:
print(ict_spec_new)

     freq size_emp nace_r2 indic_is         unit geo  num_2020 flag_2020  \
0       A     GE10       C  E_ITSP2       PC_ENT  AL       NaN       NaN   
1       A     GE10       C  E_ITSP2       PC_ENT  AT     28.74       NaN   
2       A     GE10       C  E_ITSP2       PC_ENT  BA     12.92       NaN   
3       A     GE10       C  E_ITSP2       PC_ENT  BE       NaN         u   
4       A     GE10       C  E_ITSP2       PC_ENT  BG     15.16       NaN   
...   ...      ...     ...      ...          ...  ..       ...       ...   
3338    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  SE       NaN       NaN   
3339    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  SI       NaN       NaN   
3340    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  SK       NaN       NaN   
3341    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  TR       NaN       NaN   
3342    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  UK       NaN       NaN   

      num_2022 flag_2022  num_2024 flag_2024  
0        11.22         b       NaN      

In [15]:
ict_spec_new.to_csv('ict_spec.csv', index = False)

### Education level

https://ec.europa.eu/eurostat/databrowser/view/edat_lfs_9910$defaultview/default/table?lang=en

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

educ = pd.read_csv(path + 'estat_edat_lfs_9910.tsv/estat_edat_lfs_9910.tsv', sep='\t')
print(educ)

       freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD 2008  2009     2010   \
0                               A,PC,A,ED0-2,Y15-24,F,AT   : u   : u      : u   
1                               A,PC,A,ED0-2,Y15-24,F,BA    :     :        :    
2                               A,PC,A,ED0-2,Y15-24,F,BE  : bu   : u      : u   
3                               A,PC,A,ED0-2,Y15-24,F,BG   : u   : u     : bu   
4                               A,PC,A,ED0-2,Y15-24,F,CH   : u   : u  68.1 bu   
...                                                  ...   ...   ...      ...   
241725                          A,PC,U,ED5-8,Y55-74,T,SE   : u   : u      : u   
241726                          A,PC,U,ED5-8,Y55-74,T,SI    :     :        :    
241727                          A,PC,U,ED5-8,Y55-74,T,SK   : u    :        :    
241728                          A,PC,U,ED5-8,Y55-74,T,TR    :    : u      : u   
241729                          A,PC,U,ED5-8,Y55-74,T,UK  : bu   : u     : bu   

         2011    2012    20

In [5]:
educ_split = educ[r'freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD'].str.split(',', expand=True)
educ = pd.concat([educ, educ_split], axis=1)
educ.rename(columns={0: 'freq', 1: 'unit', 2: 'nace_r2',  3: 'isced11',  4: 'age', 5: 'sex', 6:  'geo_TIME_PERIOD'}, inplace=True)
educ = educ.drop(r'freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD', axis=1)
educ.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
educ.columns = educ.columns.str.strip()
educ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241730 entries, 0 to 241729
Data columns (total 24 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   2008     241730 non-null  object
 1   2009     241730 non-null  object
 2   2010     241730 non-null  object
 3   2011     241730 non-null  object
 4   2012     241730 non-null  object
 5   2013     241730 non-null  object
 6   2014     241730 non-null  object
 7   2015     241730 non-null  object
 8   2016     241730 non-null  object
 9   2017     241730 non-null  object
 10  2018     241730 non-null  object
 11  2019     241730 non-null  object
 12  2020     241730 non-null  object
 13  2021     241730 non-null  object
 14  2022     241730 non-null  object
 15  2023     241730 non-null  object
 16  2024     241730 non-null  object
 17  freq     241730 non-null  object
 18  unit     241730 non-null  object
 19  nace_r2  241730 non-null  object
 20  isced11  241730 non-null  object
 21  age      2

In [6]:
years_take = ['2020', '2021', '2022', '2023', '2024']
data_temp  = educ.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
educ_new = data_temp
print(educ_new)

        2008 2009     2010    2011    2012    2013     2014    2015    2016  \
0        : u  : u      : u     : u     : u     : u     : bu     : u     : u   
1         :    :        :       :       :       :        :       :       :    
2       : bu  : u      : u     : u     : u     : u     : bu     : u     : u   
3        : u  : u     : bu    : bu     : u     : u     : bu     : u     : u   
4        : u  : u  68.1 bu  60.4 u  46.9 u  58.9 u  61.0 bu  36.8 u  33.1 u   
...      ...  ...      ...     ...     ...     ...      ...     ...     ...   
241725   : u  : u      : u     : u     : u     : u     : bu     : u     : u   
241726    :    :        :       :       :       :        :       :       :    
241727   : u   :        :     : bu     : u      :      : bu     : u     : u   
241728    :   : u      : u  72.4 u  74.7 u  36.0 u       :      : u     : u   
241729  : bu  : u     : bu    : bu     : u     : u     : bu     : u     : u   

          2017  ... num_2020 flag_2020 num_2021 fla

In [8]:
educ_new.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
educ_new.drop(years_take, axis='columns', inplace=True)
educ_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241730 entries, 0 to 241729
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   2013       241730 non-null  object 
 1   freq       241730 non-null  object 
 2   unit       241730 non-null  object 
 3   nace_r2    241730 non-null  object 
 4   isced11    241730 non-null  object 
 5   age        241730 non-null  object 
 6   sex        241730 non-null  object 
 7   geo        241730 non-null  object 
 8   num_2020   114497 non-null  float64
 9   flag_2020  241730 non-null  object 
 10  num_2021   153586 non-null  float64
 11  flag_2021  241730 non-null  object 
 12  num_2022   154677 non-null  float64
 13  flag_2022  241730 non-null  object 
 14  num_2023   151890 non-null  float64
 15  flag_2023  241730 non-null  object 
 16  num_2024   151934 non-null  float64
 17  flag_2024  241730 non-null  object 
dtypes: float64(5), object(13)
memory usage: 33.2+ MB


In [10]:
print(educ_new)

       freq unit nace_r2 isced11     age sex geo  num_2020 flag_2020  \
0         A   PC       A   ED0-2  Y15-24   F  AT       NaN         u   
1         A   PC       A   ED0-2  Y15-24   F  BA       NaN       NaN   
2         A   PC       A   ED0-2  Y15-24   F  BE       NaN         u   
3         A   PC       A   ED0-2  Y15-24   F  BG       NaN         u   
4         A   PC       A   ED0-2  Y15-24   F  CH       NaN         u   
...     ...  ...     ...     ...     ...  ..  ..       ...       ...   
241725    A   PC       U   ED5-8  Y55-74   T  SE       NaN         u   
241726    A   PC       U   ED5-8  Y55-74   T  SI       NaN       NaN   
241727    A   PC       U   ED5-8  Y55-74   T  SK       NaN       NaN   
241728    A   PC       U   ED5-8  Y55-74   T  TR       NaN         u   
241729    A   PC       U   ED5-8  Y55-74   T  UK       NaN       NaN   

        num_2021 flag_2021  num_2022 flag_2022  num_2023 flag_2023  num_2024  \
0            NaN        bu       NaN         u       Na

In [11]:
educ_new.to_csv('educ.csv', index = False)

### Occupation 
https://ec.europa.eu/eurostat/databrowser/view/lfsa_eisn2$defaultview/default/table?lang=en

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

df = pd.read_csv(path + 'estat_lfsa_eisn2.tsv/estat_lfsa_eisn2.tsv', sep='\t')
print(df)

      freq,age,sex,nace_r2,isco08,unit,geo\TIME_PERIOD   2008   2009   2010   \
0                          A,Y20-64,F,A,NRP,THS_PER,CH    : bu    : u   : bu   
1                          A,Y20-64,F,A,NRP,THS_PER,DE    : bu    : u   : bu   
2                          A,Y20-64,F,A,NRP,THS_PER,DK      :      :      :    
3                        A,Y20-64,F,A,NRP,THS_PER,EA20    : bu    : u    : u   
4                   A,Y20-64,F,A,NRP,THS_PER,EU27_2020    : bu    : u    : u   
...                                                ...     ...    ...    ...   
54796                    A,Y_GE15,T,U,TOTAL,THS_PER,SE    : bu    : u  1.4 u   
54797                    A,Y_GE15,T,U,TOTAL,THS_PER,SI    : bu     :      :    
54798                    A,Y_GE15,T,U,TOTAL,THS_PER,SK    : bu   : bu    : u   
54799                    A,Y_GE15,T,U,TOTAL,THS_PER,TR      :    5.3    3.7    
54800                    A,Y_GE15,T,U,TOTAL,THS_PER,UK  12.2 b  43.4   41.0    

       2011   2012   2013    2014   201

In [24]:
df_split = df[r'freq,age,sex,nace_r2,isco08,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
df = pd.concat([df, df_split], axis=1)
df.rename(columns={0: 'freq', 1: 'age', 2: 'sex',  3: 'nace_r2',  4: 'isco08', 5: 'unit', 6:  'geo_TIME_PERIOD'}, inplace=True)
df = df.drop(r'freq,age,sex,nace_r2,isco08,unit,geo\TIME_PERIOD', axis=1)
df.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
df.columns = df.columns.str.strip()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54801 entries, 0 to 54800
Data columns (total 24 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   2008     54801 non-null  object
 1   2009     54801 non-null  object
 2   2010     54801 non-null  object
 3   2011     54801 non-null  object
 4   2012     54801 non-null  object
 5   2013     54801 non-null  object
 6   2014     54801 non-null  object
 7   2015     54801 non-null  object
 8   2016     54801 non-null  object
 9   2017     54801 non-null  object
 10  2018     54801 non-null  object
 11  2019     54801 non-null  object
 12  2020     54801 non-null  object
 13  2021     54801 non-null  object
 14  2022     54801 non-null  object
 15  2023     54801 non-null  object
 16  2024     54801 non-null  object
 17  freq     54801 non-null  object
 18  age      54801 non-null  object
 19  sex      54801 non-null  object
 20  nace_r2  54801 non-null  object
 21  isco08   54801 non-null  object
 22

In [27]:
years_take = ['2020', '2021', '2022', '2023', '2024']
data_temp  = df.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
df_new = data_temp
print(df_new)

         2008   2009   2010   2011   2012   2013    2014   2015   2016   2017  \
0        : bu    : u   : bu   : bu    : u    : u     : u    : u    : u    : u   
1        : bu    : u   : bu   : bu     :     : u      :      :      :     : u   
2          :      :      :      :      :      :      : u     :      :      :    
3        : bu    : u    : u   : bu    : u    : u     : u    : u    : u    : u   
4        : bu    : u    : u   : bu    : u    : u     : u    : u    : u    : u   
...       ...    ...    ...    ...    ...    ...     ...    ...    ...    ...   
54796    : bu    : u  1.4 u  1.6 u  1.9 u   2.5    1.6 u    : u    : u  1.3 u   
54797    : bu     :      :      :      :      :       :      :      :      :    
54798    : bu   : bu    : u   : bu    : u    : u     : u    : u    : u    : u   
54799      :    5.3    3.7    4.7    5.2    5.5   2.3 bu   4.0    6.1    9.2    
54800  12.2 b  43.4   41.0   50.3   40.8   42.0    43.1   39.3   49.8   44.9    

       ... num_2020 flag_20

In [28]:
df_new.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
df_new.drop(years_take, axis='columns', inplace=True)
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54801 entries, 0 to 54800
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   freq       54801 non-null  object 
 1   age        54801 non-null  object 
 2   sex        54801 non-null  object 
 3   nace_r2    54801 non-null  object 
 4   isco08     54801 non-null  object 
 5   unit       54801 non-null  object 
 6   geo        54801 non-null  object 
 7   num_2020   28535 non-null  float64
 8   flag_2020  54801 non-null  object 
 9   num_2021   28613 non-null  float64
 10  flag_2021  54801 non-null  object 
 11  num_2022   28797 non-null  float64
 12  flag_2022  54801 non-null  object 
 13  num_2023   28246 non-null  float64
 14  flag_2023  54801 non-null  object 
 15  num_2024   28303 non-null  float64
 16  flag_2024  54801 non-null  object 
dtypes: float64(5), object(12)
memory usage: 7.1+ MB


In [29]:
print(df_new)

      freq     age sex nace_r2 isco08     unit        geo  num_2020 flag_2020  \
0        A  Y20-64   F       A    NRP  THS_PER         CH       NaN         u   
1        A  Y20-64   F       A    NRP  THS_PER         DE       NaN        bu   
2        A  Y20-64   F       A    NRP  THS_PER         DK       NaN       NaN   
3        A  Y20-64   F       A    NRP  THS_PER       EA20       NaN         u   
4        A  Y20-64   F       A    NRP  THS_PER  EU27_2020       NaN         u   
...    ...     ...  ..     ...    ...      ...        ...       ...       ...   
54796    A  Y_GE15   T       U  TOTAL  THS_PER         SE       NaN         u   
54797    A  Y_GE15   T       U  TOTAL  THS_PER         SI       NaN         u   
54798    A  Y_GE15   T       U  TOTAL  THS_PER         SK       NaN         u   
54799    A  Y_GE15   T       U  TOTAL  THS_PER         TR       6.8       NaN   
54800    A  Y_GE15   T       U  TOTAL  THS_PER         UK       NaN       NaN   

       num_2021 flag_2021  

In [30]:
df_new.to_csv('occup.csv', index = False)

### Unemployment rate 

https://ec.europa.eu/eurostat/databrowser/view/tps00203/default/table?lang=en&category=t_labour.t_employ.t_lfsi.t_une

In [16]:
# path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

# df = pd.read_csv(path + 'estat_tps00203.tsv/estat_tps00203.tsv', sep='\t')
# print(df)

df = eurostat.get_data_df('tps00203')
print(df)

    freq     age     unit sex geo\TIME_PERIOD    2013    2014    2015    2016  \
0      A  Y15-74   PC_ACT   T              AT     5.7     6.0     6.1     6.5   
1      A  Y15-74   PC_ACT   T              BA     NaN     NaN     NaN     NaN   
2      A  Y15-74   PC_ACT   T              BE     8.6     8.7     8.7     7.9   
3      A  Y15-74   PC_ACT   T              BG    13.9    12.4    10.1     8.6   
4      A  Y15-74   PC_ACT   T              CH     4.8     4.9     4.8     5.0   
..   ...     ...      ...  ..             ...     ...     ...     ...     ...   
109    A  Y15-74  THS_PER   T              RS   732.0   627.0   570.0   506.0   
110    A  Y15-74  THS_PER   T              SE   414.0   414.0   391.0   371.0   
111    A  Y15-74  THS_PER   T              SI   101.0    98.0    90.0    79.0   
112    A  Y15-74  THS_PER   T              SK   394.0   366.0   323.0   274.0   
113    A  Y15-74  THS_PER   T              TR  2442.0  2843.0  3035.0  3308.0   

       2017    2018    2019

In [ ]:
# df_split = df[r'freq,age,unit,sex,geo\TIME_PERIOD'].str.split(',', expand=True)
# df = pd.concat([df, df_split], axis=1)
# df.rename(columns={0: 'freq', 1: 'age', 2: 'unit',  3: 'sex',  4:  'geo_TIME_PERIOD'}, inplace=True)
# df = df.drop(r'freq,age,unit,sex,geo\TIME_PERIOD', axis=1)
# df.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
# df.columns = df.columns.str.strip()
# df.info()

In [17]:
df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   freq    114 non-null    object 
 1   age     114 non-null    object 
 2   unit    114 non-null    object 
 3   sex     114 non-null    object 
 4   geo     114 non-null    object 
 5   2013    111 non-null    float64
 6   2014    111 non-null    float64
 7   2015    111 non-null    float64
 8   2016    111 non-null    float64
 9   2017    111 non-null    float64
 10  2018    111 non-null    float64
 11  2019    111 non-null    float64
 12  2020    111 non-null    float64
 13  2021    111 non-null    float64
 14  2022    111 non-null    float64
 15  2023    111 non-null    float64
 16  2024    111 non-null    float64
dtypes: float64(12), object(5)
memory usage: 15.3+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_10484\1019875136.py:1: SyntaxWarning: invalid escape sequence '\T'
  df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [ ]:
# years_take = ['2020', '2021', '2022', '2023', '2024']
# data_temp  = df.copy()

# pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

# for year in years_take:
#     column_series = data_temp[year].astype(str).str.strip() 

#     extracted_df = column_series.str.extract(pattern, expand=True)
#     # print(extracted_df)

#     num_col_name = f'num_{year}'
#     data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
#     # print(ain2[num_col_name])
    
#     flag_col_name = f'flag_{year}'
#     data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
#     # print(ain2[flag_col_name])
#     # print('#########################')
# # Display the resulting DataFrame
# df_new = data_temp
# print(df_new)

In [19]:
df.drop(['2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   freq    114 non-null    object 
 1   age     114 non-null    object 
 2   unit    114 non-null    object 
 3   sex     114 non-null    object 
 4   geo     114 non-null    object 
 5   2020    111 non-null    float64
 6   2021    111 non-null    float64
 7   2022    111 non-null    float64
 8   2023    111 non-null    float64
 9   2024    111 non-null    float64
dtypes: float64(5), object(5)
memory usage: 9.0+ KB


In [21]:
df.to_csv('processed data/unempl_r.csv', index = False)

### Inflation rate
https://ec.europa.eu/eurostat/databrowser/product/page/TEC00118


In [22]:
# path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

# df = pd.read_csv(path + 'estat_tec00118.tsv/estat_tec00118.tsv', sep='\t')
# print(df)

df = eurostat.get_data_df('tec00118')
print(df)

   freq       unit coicop18 geo\TIME_PERIOD  2014  2015  2016  2017  2018  \
0     A  RCH_A_AVG    TOTAL              AL   NaN   NaN   NaN   NaN   1.8   
1     A  RCH_A_AVG    TOTAL              AT   1.5   0.8   1.0   2.2   2.1   
2     A  RCH_A_AVG    TOTAL              BE   0.5   0.6   1.8   2.2   2.3   
3     A  RCH_A_AVG    TOTAL              BG  -1.6  -1.1  -1.3   1.2   2.6   
4     A  RCH_A_AVG    TOTAL              CH   0.0  -0.8  -0.5   0.6   0.9   
5     A  RCH_A_AVG    TOTAL              CY  -0.3  -1.5  -1.2   0.7   0.8   
6     A  RCH_A_AVG    TOTAL              CZ   0.4   0.3   0.7   2.5   1.9   
7     A  RCH_A_AVG    TOTAL              DE   0.8   0.7   0.4   1.7   1.9   
8     A  RCH_A_AVG    TOTAL              DK   0.3   0.2   0.0   1.1   0.7   
9     A  RCH_A_AVG    TOTAL            EA19   0.4   0.2   0.2   1.5   1.8   
10    A  RCH_A_AVG    TOTAL            EA20   0.4   0.2   0.2   1.5   1.8   
11    A  RCH_A_AVG    TOTAL            EA21   0.4   0.2   0.2   1.5   1.8   

In [23]:
df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      41 non-null     object 
 1   unit      41 non-null     object 
 2   coicop18  41 non-null     object 
 3   geo       41 non-null     object 
 4   2014      37 non-null     float64
 5   2015      37 non-null     float64
 6   2016      37 non-null     float64
 7   2017      39 non-null     float64
 8   2018      41 non-null     float64
 9   2019      41 non-null     float64
 10  2020      40 non-null     float64
 11  2021      40 non-null     float64
 12  2022      40 non-null     float64
 13  2023      40 non-null     float64
 14  2024      40 non-null     float64
 15  2025      39 non-null     float64
dtypes: float64(12), object(4)
memory usage: 5.3+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_10484\1019875136.py:1: SyntaxWarning: invalid escape sequence '\T'
  df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [25]:
df.drop(['2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      41 non-null     object 
 1   unit      41 non-null     object 
 2   coicop18  41 non-null     object 
 3   geo       41 non-null     object 
 4   2020      40 non-null     float64
 5   2021      40 non-null     float64
 6   2022      40 non-null     float64
 7   2023      40 non-null     float64
 8   2024      40 non-null     float64
 9   2025      39 non-null     float64
dtypes: float64(6), object(4)
memory usage: 3.3+ KB


In [26]:
df.to_csv('processed data/infl_r.csv', index = False)

### GDP and main components (output, expenditure and income)
https://ec.europa.eu/eurostat/databrowser/view/nama_10_pc/default/table?lang=en

In [33]:
df = eurostat.get_data_df('nama_10_pc')
print(df)

     freq                      unit na_item geo\TIME_PERIOD  1975  1976  1977  \
0       A             CLV10_EUR_HAB    B1GQ              AL   NaN   NaN   NaN   
1       A             CLV10_EUR_HAB    B1GQ              AT   NaN   NaN   NaN   
2       A             CLV10_EUR_HAB    B1GQ              BE   NaN   NaN   NaN   
3       A             CLV10_EUR_HAB    B1GQ              BG   NaN   NaN   NaN   
4       A             CLV10_EUR_HAB    B1GQ              CH   NaN   NaN   NaN   
...   ...                       ...     ...             ...   ...   ...   ...   
4486    A  PC_EU27_2020_HAB_MPPS_CP     P41              SE   NaN   NaN   NaN   
4487    A  PC_EU27_2020_HAB_MPPS_CP     P41              SI   NaN   NaN   NaN   
4488    A  PC_EU27_2020_HAB_MPPS_CP     P41              SK   NaN   NaN   NaN   
4489    A  PC_EU27_2020_HAB_MPPS_CP     P41              TR   NaN   NaN   NaN   
4490    A  PC_EU27_2020_HAB_MPPS_CP     P41              UK   NaN   NaN   NaN   

      1978  1979  1980  ...

In [34]:
df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4491 entries, 0 to 4490
Data columns (total 55 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     4491 non-null   object 
 1   unit     4491 non-null   object 
 2   na_item  4491 non-null   object 
 3   geo      4491 non-null   object 
 4   1975     243 non-null    float64
 5   1976     243 non-null    float64
 6   1977     243 non-null    float64
 7   1978     243 non-null    float64
 8   1979     243 non-null    float64
 9   1980     315 non-null    float64
 10  1981     324 non-null    float64
 11  1982     324 non-null    float64
 12  1983     324 non-null    float64
 13  1984     324 non-null    float64
 14  1985     324 non-null    float64
 15  1986     324 non-null    float64
 16  1987     324 non-null    float64
 17  1988     324 non-null    float64
 18  1989     324 non-null    float64
 19  1990     324 non-null    float64
 20  1991     396 non-null    float64
 21  1992     405 n

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_10484\1019875136.py:1: SyntaxWarning: invalid escape sequence '\T'
  df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [35]:
columns_to_drop = df.columns[4:49]
df.drop(columns=columns_to_drop, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4491 entries, 0 to 4490
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     4491 non-null   object 
 1   unit     4491 non-null   object 
 2   na_item  4491 non-null   object 
 3   geo      4491 non-null   object 
 4   2020     4383 non-null   float64
 5   2021     4383 non-null   float64
 6   2022     4374 non-null   float64
 7   2023     4266 non-null   float64
 8   2024     4245 non-null   float64
 9   2025     162 non-null    float64
dtypes: float64(6), object(4)
memory usage: 351.0+ KB


In [36]:
df.to_csv('processed data/gdp.csv', index = False)

### GDP and main components per capita
https://ec.europa.eu/eurostat/databrowser/view/nama_10_pc/default/table?lang=en&category=na10.nama10.nama_10_ma

In [37]:
df = eurostat.get_data_df('nama_10_pc')
print(df)

     freq                      unit na_item geo\TIME_PERIOD  1975  1976  1977  \
0       A             CLV10_EUR_HAB    B1GQ              AL   NaN   NaN   NaN   
1       A             CLV10_EUR_HAB    B1GQ              AT   NaN   NaN   NaN   
2       A             CLV10_EUR_HAB    B1GQ              BE   NaN   NaN   NaN   
3       A             CLV10_EUR_HAB    B1GQ              BG   NaN   NaN   NaN   
4       A             CLV10_EUR_HAB    B1GQ              CH   NaN   NaN   NaN   
...   ...                       ...     ...             ...   ...   ...   ...   
4486    A  PC_EU27_2020_HAB_MPPS_CP     P41              SE   NaN   NaN   NaN   
4487    A  PC_EU27_2020_HAB_MPPS_CP     P41              SI   NaN   NaN   NaN   
4488    A  PC_EU27_2020_HAB_MPPS_CP     P41              SK   NaN   NaN   NaN   
4489    A  PC_EU27_2020_HAB_MPPS_CP     P41              TR   NaN   NaN   NaN   
4490    A  PC_EU27_2020_HAB_MPPS_CP     P41              UK   NaN   NaN   NaN   

      1978  1979  1980  ...

In [38]:
df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
columns_to_drop = df.columns[4:49]
df.drop(columns=columns_to_drop, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4491 entries, 0 to 4490
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     4491 non-null   object 
 1   unit     4491 non-null   object 
 2   na_item  4491 non-null   object 
 3   geo      4491 non-null   object 
 4   2020     4383 non-null   float64
 5   2021     4383 non-null   float64
 6   2022     4374 non-null   float64
 7   2023     4266 non-null   float64
 8   2024     4245 non-null   float64
 9   2025     162 non-null    float64
dtypes: float64(6), object(4)
memory usage: 351.0+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_10484\3893668518.py:1: SyntaxWarning: invalid escape sequence '\T'
  df.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [40]:
df.to_csv('processed data/gdp_per_cap.csv', index = False)

### Productivity 
https://ec.europa.eu/eurostat/databrowser/product/page/SBS_SC_OVW

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

df = pd.read_csv(path + 'estat_sbs_sc_ovw.tsv/estat_sbs_sc_ovw.tsv', sep='\t')
print(df)

        freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD    2021     2022   \
0                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,AL    5.11     5.07    
1                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,AT   49.73    54.31    
2                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BA    7.26     8.89    
3                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BE   45.15    53.13    
4                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BG   12.48    11.54    
...                                                 ...      ...      ...   
1406053                       A,WAGE_MEUR,S960,TOTAL,RO  199.03   211.74    
1406054                       A,WAGE_MEUR,S960,TOTAL,RS  43.30 b   48.03    
1406055                       A,WAGE_MEUR,S960,TOTAL,SE  863.44   891.76    
1406056                       A,WAGE_MEUR,S960,TOTAL,SI   65.60    71.97    
1406057                       A,WAGE_MEUR,S960,TOTAL,SK   46.87    56.30    

           2023   
0          5.79   
1         55.31   
2         10.07   

In [58]:
df_split = df[r'freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD'].str.split(',', expand=True)
df = pd.concat([df, df_split], axis=1)
df.rename(columns={0: 'freq', 1: 'indic_sbs',  2: 'nace_r2',  3: 'size_emp', 4: 'geo_TIME_PERIOD'}, inplace=True)
df = df.drop(r'freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD', axis=1)
df.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
df.columns = df.columns.str.strip()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1406058 entries, 0 to 1406057
Data columns (total 8 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   2021       1406058 non-null  object
 1   2022       1406058 non-null  object
 2   2023       1406058 non-null  object
 3   freq       1406058 non-null  object
 4   indic_sbs  1406058 non-null  object
 5   nace_r2    1406058 non-null  object
 6   size_emp   1406058 non-null  object
 7   geo        1406058 non-null  object
dtypes: object(8)
memory usage: 85.8+ MB


In [59]:
years_take = ['2021', '2022', '2023']
data_temp  = df.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
df_new = data_temp
print(df_new)

            2021     2022     2023 freq              indic_sbs nace_r2  \
0          5.11     5.07     5.79     A  AVG_EXPN_SAL_BEN_TEUR       B   
1         49.73    54.31    55.31     A  AVG_EXPN_SAL_BEN_TEUR       B   
2          7.26     8.89    10.07     A  AVG_EXPN_SAL_BEN_TEUR       B   
3         45.15    53.13    51.45     A  AVG_EXPN_SAL_BEN_TEUR       B   
4         12.48    11.54    12.77     A  AVG_EXPN_SAL_BEN_TEUR       B   
...          ...      ...      ...  ...                    ...     ...   
1406053  199.03   211.74   256.91     A              WAGE_MEUR    S960   
1406054  43.30 b   48.03    57.22     A              WAGE_MEUR    S960   
1406055  863.44   891.76   805.37     A              WAGE_MEUR    S960   
1406056   65.60    71.97    81.64     A              WAGE_MEUR    S960   
1406057   46.87    56.30    65.11     A              WAGE_MEUR    S960   

        size_emp geo  num_2021 flag_2021  num_2022 flag_2022  num_2023  \
0            0-9  AL      5.11       

In [60]:
df_new.to_csv('prodct.csv', index = False)

### Firm size

https://ec.europa.eu/eurostat/databrowser/product/view/sbs_sc_ovw$dv_1482?category=cult.cult_ent

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

df = pd.read_csv(path + 'estat_sbs_sc_ovw.tsv/estat_sbs_sc_ovw.tsv', sep='\t')
print(df)

        freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD    2021     2022   \
0                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,AL    5.11     5.07    
1                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,AT   49.73    54.31    
2                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BA    7.26     8.89    
3                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BE   45.15    53.13    
4                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BG   12.48    11.54    
...                                                 ...      ...      ...   
1406053                       A,WAGE_MEUR,S960,TOTAL,RO  199.03   211.74    
1406054                       A,WAGE_MEUR,S960,TOTAL,RS  43.30 b   48.03    
1406055                       A,WAGE_MEUR,S960,TOTAL,SE  863.44   891.76    
1406056                       A,WAGE_MEUR,S960,TOTAL,SI   65.60    71.97    
1406057                       A,WAGE_MEUR,S960,TOTAL,SK   46.87    56.30    

           2023   
0          5.79   
1         55.31   
2         10.07   

In [62]:
df_split = df[r'freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD'].str.split(',', expand=True)
df = pd.concat([df, df_split], axis=1)
df.rename(columns={0: 'freq', 1: 'indic_sbs',  2: 'nace_r2',  3: 'size_emp', 4: 'geo_TIME_PERIOD'}, inplace=True)
df = df.drop(r'freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD', axis=1)
df.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
df.columns = df.columns.str.strip()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1406058 entries, 0 to 1406057
Data columns (total 8 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   2021       1406058 non-null  object
 1   2022       1406058 non-null  object
 2   2023       1406058 non-null  object
 3   freq       1406058 non-null  object
 4   indic_sbs  1406058 non-null  object
 5   nace_r2    1406058 non-null  object
 6   size_emp   1406058 non-null  object
 7   geo        1406058 non-null  object
dtypes: object(8)
memory usage: 85.8+ MB


In [63]:
years_take = ['2021', '2022', '2023']
data_temp  = df.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    flag_col_name = f'flag_{year}'
    data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
df_new = data_temp
print(df_new)

            2021     2022     2023 freq              indic_sbs nace_r2  \
0          5.11     5.07     5.79     A  AVG_EXPN_SAL_BEN_TEUR       B   
1         49.73    54.31    55.31     A  AVG_EXPN_SAL_BEN_TEUR       B   
2          7.26     8.89    10.07     A  AVG_EXPN_SAL_BEN_TEUR       B   
3         45.15    53.13    51.45     A  AVG_EXPN_SAL_BEN_TEUR       B   
4         12.48    11.54    12.77     A  AVG_EXPN_SAL_BEN_TEUR       B   
...          ...      ...      ...  ...                    ...     ...   
1406053  199.03   211.74   256.91     A              WAGE_MEUR    S960   
1406054  43.30 b   48.03    57.22     A              WAGE_MEUR    S960   
1406055  863.44   891.76   805.37     A              WAGE_MEUR    S960   
1406056   65.60    71.97    81.64     A              WAGE_MEUR    S960   
1406057   46.87    56.30    65.11     A              WAGE_MEUR    S960   

        size_emp geo  num_2021 flag_2021  num_2022 flag_2022  num_2023  \
0            0-9  AL      5.11       

In [64]:
columns_to_drop = df_new.columns[0:3]
df_new.drop(columns=columns_to_drop, inplace=True)
# df_new.drop(years_take, axis='columns', inplace=True)
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1406058 entries, 0 to 1406057
Data columns (total 11 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   freq       1406058 non-null  object 
 1   indic_sbs  1406058 non-null  object 
 2   nace_r2    1406058 non-null  object 
 3   size_emp   1406058 non-null  object 
 4   geo        1406058 non-null  object 
 5   num_2021   1021169 non-null  float64
 6   flag_2021  1406058 non-null  object 
 7   num_2022   1051992 non-null  float64
 8   flag_2022  1406058 non-null  object 
 9   num_2023   1051803 non-null  float64
 10  flag_2023  1406058 non-null  object 
dtypes: float64(3), object(8)
memory usage: 118.0+ MB


In [65]:
df_new.to_csv('firm_s.csv', index = False)